# Sifting Logs

This notebook explores the logs generated by the Fortresses (`zelda.txt` and `castle2.txt`) to surface the most compelling narrative moments.  

## Key Ideas
- **Patterns**: Identify and apply story patterns to structure the sifting.  
- **Weights**: Assign weights to actions, emphasizing their narrative significance.  
- **Coverage**: Ensure coverage across the logs so that important variations aren’t overlooked.  

#### Story sifting patterns:
- Predator–Prey / Conflict Patterns (based on chased)
- Gathering items (based on took)
- Finding treasure 


In [418]:
import random
from collections import Counter

In [ ]:
class LogSifter:
    # Class that encompasses all the sifting pattern functions
    def __init__(self, log_file:str):
        self.log_file = log_file
        self.logs = self.load_logs()
        self.strip_log_file()        

        self.action_weight_mapping = {
            'moved': 0.5,
            'took': 3,
            'chased': 3,
            'died': 6,
            'pushed': 4,
            'added': 5,
            'blocked': 2
        }


        # print(self.predator_prey_pattern())
        # print(self.gathering_pattern())
        # print(self.finding_treasure_pattern())

        # print(self.sifted_logs)

    def load_logs(self):
        ''' Loads the logs from the specified log file '''
        with open(self.log_file, 'r') as f:
            logs = f.readlines()
        return logs[3:] # Strip the first three lines 

    def strip_log_file(self):
        ''' Strips the log file of \n characters '''
        self.logs = [log.strip('\n') for log in self.logs]

    def predator_prey_pattern(self):
        ''' Identifies predator-prey relationships in the logs '''

        selected_logs = []
        for i in range(len(self.logs)):
            # separate index from log
            split_log = self.logs[i].split(' ')
            split_log_index = split_log.pop(0)

            if 'chased' in split_log and ' '.join(split_log) not in selected_logs:
                chaser = split_log[0]
                prey = split_log[2]

                # check if chaser catches the prey with "took"
                found_took = False
                for j in range(i + 1, len(self.logs)):
                    split_next_log = self.logs[j].split(' ')
                    split_next_log_index = split_next_log.pop(0)
                    if 'took' in split_next_log and split_next_log[0] == chaser and split_next_log[2] == prey:
                        combined_log = f"{split_log_index} {' '.join(split_log)}"
                        selected_logs.append(combined_log)
                        combined_log = f"{split_next_log_index} {' '.join(split_next_log)}"
                        selected_logs.append(combined_log)
                        found_took = True
                        break
                
                if not found_took:
                    combined_log = f"{split_log_index} {' '.join(split_log)}"
                    selected_logs.append(combined_log)
                break
        return selected_logs

    def gathering_pattern(self):
        # pattern where if one single entity takes many things, note them all
        selected_logs = []
        for i in range(len(self.logs)):
            split_log = self.logs[i].split(' ')
            split_log_index = split_log.pop(0)

            if 'took' in split_log and ' '.join(split_log) not in selected_logs:
                taker = split_log[0]
                taker_logs = [split_log]
                taker_log_index = [split_log_index]

                # check the rest of the log to find other items the taker has taken
                for j in range(i + 1, len(self.logs)):
                    split_next_log = self.logs[j].split(' ')
                    split_next_log_index = split_next_log.pop(0)
                    if 'took' in split_next_log and split_next_log[0] == taker:
                        if ' '.join(split_next_log) not in selected_logs:
                            taker_logs.append(split_next_log)
                            taker_log_index.append(split_next_log_index)

                # If the taker has taken multiple items, we need to condense the logs
                if len(taker_logs) > 1:
                    for k, log in enumerate(taker_logs):
                        combined_log = f"{taker_log_index[k]} {' '.join(log)}"
                        selected_logs.append(combined_log)
                        break
        return selected_logs

    def finding_treasure_pattern(self):
        # pattern if one entity moves around a lot but takes only one thing

        selected_logs = []
        for i in range(len(self.logs)):
            split_log = self.logs[i].split(' ')
            split_log_index = split_log.pop(0)

            if 'moved' in split_log and ' '.join(split_log) not in selected_logs:
                mover = split_log[0]
                mover_logs = [split_log]
                mover_log_index = [split_log_index]
                took_counter = 0

                # check the rest of the log to find other items the mover has taken
                for j in range(i + 1, len(self.logs)):
                    split_next_log = self.logs[j].split(' ')
                    split_next_log_index = split_next_log.pop(0)

                    # add any other movement the mover makes
                    if 'moved' in split_next_log and split_next_log[0] == mover:
                        mover_logs.append(split_next_log)
                        mover_log_index.append(split_next_log_index)

                    if 'took' in split_next_log and split_next_log[0] == mover:
                        took_counter += 1
                        if ' '.join(split_next_log) not in selected_logs:
                            mover_logs.append(split_next_log)
                            mover_log_index.append(split_next_log_index)
                            break
                    

                # If the mover has taken only one item after moving, condense the logs
                if len(mover_logs) > 1 and took_counter == 1:
                    move_max = random.randint(1, 3)  # allow 1–3 moved actions total
                    move_count = 0

                    for k, log in enumerate(mover_logs):
                        if 'moved' in log and move_count < move_max:
                            # combine index and log and then append to selected log with whitespace alone
                            combined_log = f"{mover_log_index[k]} {' '.join(log)}"
                            selected_logs.append(combined_log)

                            move_count += 1
                        elif 'took' in log:
                            combined_log = f"{mover_log_index[k]} {' '.join(log)}"
                            selected_logs.append(combined_log)
                    
        return selected_logs

    def sift_logs(self):
        # gather a total of logs using a mix of patterns + weights + random indexes
        total_logs = 30
        gathered_logs = []  # Use list to maintain order
        log_count = 0
        used_logs = set()  # Track which log contents we've used to avoid duplicates

        pattern_prob = 0.5
        weighted_prob = 0.3
        random_prob = 0.2

        while len(gathered_logs) < total_logs:
            selected_log = None
            pattern_logs = []
            
            prob = random.random()
            # print(prob)
            if prob < random_prob:
                print("selecting randomly")
                available_logs = [log for log in self.logs if log not in used_logs]
                if available_logs:
                    selected_log = random.choice(available_logs)


            elif prob < weighted_prob + random_prob:
                print("selecting based on weights")
                available_logs = [log for log in self.logs if log not in used_logs]
                if available_logs:
                    actions = [log.split()[2] for log in available_logs]
                    counts = Counter(actions)

                    # weight = inverse frequency
                    weights = [1 / counts[action] for action in actions]

                    # pick based on rarity
                    selected_log = random.choices(available_logs, weights=weights, k=1)[0]

            else:
                print("selecting based on patterns")
                pattern_prob = random.randint(0,2)

                if pattern_prob == 0:
                    print("selecting predator-prey pattern")
                    pattern_logs = self.predator_prey_pattern()
                elif pattern_prob == 1:
                    print("selecting gathering pattern")
                    pattern_logs = self.gathering_pattern()
                elif pattern_prob == 2:
                    print("selecting finding treasure pattern")
                    pattern_logs = self.finding_treasure_pattern()
            

            print("selected logs:")

            if pattern_logs:
                if len(pattern_logs) + len(gathered_logs) < total_logs:
                    for log in pattern_logs:
                        if log not in used_logs:
                            gathered_logs.append(log)
                            used_logs.add(log)
                            print(log)

            elif selected_log and selected_log not in used_logs:
                print(selected_log)
                gathered_logs.append(selected_log)
                used_logs.add(selected_log)

        gathered_logs.sort(key=lambda log: int(log.split()[0].strip("<>")))

        return gathered_logs

In [413]:
# Test
log_sifter = LogSifter('../logs/castle2.txt')
# log_sifter = LogSifter('../logs/zelda.txt')

sifted_logs = log_sifter.sift_logs()

for log in sifted_logs:
    print(log)

# print(log_sifter.sift_logs())
print(len(log_sifter.sift_logs()))

selecting based on patterns
selecting predator-prey pattern
selected logs:
selecting based on weights
selected logs:
<43> [T.73ab] took [$.db0a]
selecting based on patterns
selecting finding treasure pattern
selected logs:
selecting based on patterns
selecting finding treasure pattern
selected logs:
selecting based on weights
selected logs:
<10> [G.3305] died
selecting based on weights
selected logs:
<9> [T.d89f] blocked by [=.23f6]
selecting based on weights
selected logs:
<27> [T.73ab] blocked by [=.0cc4]
selecting based on patterns
selecting gathering pattern
selected logs:
<10> [T.73ab] took [$.1a6e]
<13> [T.73ab] took [$.6d7b]
<15> [K.fc7c] took [T.d89f]
<21> [T.73ab] took [$.9fe4]
<25> [T.73ab] took [$.1b0e]
<29> [T.73ab] took [$.0bdf]
<38> [T.73ab] took [$.6666]
<41> [T.73ab] took [$.6924]
selecting randomly
selected logs:
<4> [T.73ab] moved to 4,5
selecting based on patterns
selecting finding treasure pattern
selected logs:
selecting randomly
selected logs:
<11> [T.d89f] moved 

In [414]:
# dump sifted logs output into txt file in /sifted_logs


log_sifter = LogSifter('../logs/castle2.txt')

sifted_logs = log_sifter.sift_logs()
with open('../sifted_logs/sifted_castle2.txt', 'w') as f:
    for log in sifted_logs:
        f.write(f"{log}\n")




selecting based on patterns
selecting gathering pattern
selected logs:
<10> [T.73ab] took [$.1a6e]
<13> [T.73ab] took [$.6d7b]
<15> [K.fc7c] took [T.d89f]
<21> [T.73ab] took [$.9fe4]
<25> [T.73ab] took [$.1b0e]
<29> [T.73ab] took [$.0bdf]
<38> [T.73ab] took [$.6666]
<41> [T.73ab] took [$.6924]
<43> [T.73ab] took [$.db0a]
selecting based on patterns
selecting gathering pattern
selected logs:
selecting based on weights
selected logs:
<6> [K.fc7c] moved towards [T.d89f]
selecting based on weights
selected logs:
<9> [T.76a3] moved to 6,6
selecting based on weights
selected logs:
<10> [T.76a3] moved to 5,6
selecting based on patterns
selecting predator-prey pattern
selected logs:
selecting based on patterns
selecting predator-prey pattern
selected logs:
selecting based on patterns
selecting finding treasure pattern
selected logs:
selecting randomly
selected logs:
<48> [K.fc7c] moved towards [T.73ab]
selecting based on weights
selected logs:
<10> [G.3305] died
selecting based on weights
sele

In [417]:
log_sifter = LogSifter('../logs/zelda.txt')
sifted_logs = log_sifter.sift_logs()
with open('../sifted_logs/sifted_zelda.txt', 'w') as f:
    for log in sifted_logs:
        f.write(f"{log}\n")

selecting based on patterns
selecting predator-prey pattern
selected logs:
<15> [L.6d7b] chased [$.3305]
selecting based on weights
selected logs:
<77> [G.4709] moved to 9,2
selecting randomly
selected logs:
<11> [G.6924] moved to 11,5
selecting based on weights
selected logs:
<26> [B.5b49] moved to 10,1
selecting based on patterns
selecting finding treasure pattern
selected logs:
<6> [B.5b49] moved to 12,1
<22> [B.5b49] took [L.6d7b]
selecting based on patterns
selecting gathering pattern
selected logs:
selecting based on weights
selected logs:
<27> [G.6924] pushed [*.9fe4]
selecting based on weights
selected logs:
<19> [B.5b49] chased [L.6d7b]
selecting randomly
selected logs:
<75> [G.4709] moved to 9,3
selecting based on patterns
selecting finding treasure pattern
selected logs:
selecting based on patterns
selecting finding treasure pattern
selected logs:
selecting based on patterns
selecting gathering pattern
selected logs:
selecting based on patterns
selecting predator-prey patter

In [381]:
# Analyze the distribution of actions in sifted logs
from collections import Counter

log_sifter = LogSifter('../logs/zelda.txt')
sifted_logs = log_sifter.sift_logs()

# Extract actions from logs
actions = [log.split(' ')[1] if len(log.split(' ')) > 1 else 'unknown' for log in sifted_logs]
action_counts = Counter(actions)

print("Action distribution in sifted zelda logs:")
for action, count in action_counts.most_common():
    percentage = (count / len(sifted_logs)) * 100
    print(f"{action}: {count} ({percentage:.1f}%)")

print(f"\nTotal logs: {len(sifted_logs)}")
print(f"Move actions: {action_counts.get('moved', 0)} out of {len(sifted_logs)} ({(action_counts.get('moved', 0)/len(sifted_logs)*100):.1f}%)")

selecting based on patterns
selecting finding treasure pattern
selecting based on patterns
selecting predator-prey pattern
selecting randomly
selecting based on patterns
selecting predator-prey pattern
selecting based on weights
selecting based on weights
selecting based on patterns
selecting gathering pattern
selecting based on patterns
selecting finding treasure pattern
selecting randomly
selecting randomly
selecting based on weights
selecting based on weights
selecting randomly
selecting randomly
selecting based on patterns
selecting finding treasure pattern
selecting based on weights
selecting based on patterns
selecting predator-prey pattern
selecting randomly
selecting based on patterns
selecting gathering pattern
selecting based on patterns
selecting predator-prey pattern
selecting based on patterns
selecting finding treasure pattern
selecting based on patterns
selecting gathering pattern
selecting randomly
selecting based on patterns
selecting finding treasure pattern
selecting